In [45]:
%pip install opencv-python
%pip install gdown
%pip install matplotlib
%pip install scikit-learn


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [46]:
from pathlib import Path
import shutil

Path("data.zip").unlink(missing_ok=True)     
shutil.rmtree("data", ignore_errors=True)    

In [47]:
import os
import zipfile
import gdown
from pathlib import Path

FILE_ID = "1a4w4HfwvyzmdIF56TmWL5-z8of0ghE3m"
zip_path = Path("data.zip")
extract_dir = Path("data")

if not extract_dir.exists():
    zip_path.unlink(missing_ok=True)               
    gdown.download(id=FILE_ID, output=str(zip_path), quiet=False)

    if not zipfile.is_zipfile(zip_path):            
        print(open(zip_path, "rb").read(300))       
        raise RuntimeError("Downloaded file is not a zip (see the text above).")

    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)

print(sorted(os.listdir(extract_dir)))

Downloading...
From (original): https://drive.google.com/uc?id=1a4w4HfwvyzmdIF56TmWL5-z8of0ghE3m
From (redirected): https://drive.google.com/uc?id=1a4w4HfwvyzmdIF56TmWL5-z8of0ghE3m&confirm=t&uuid=d26b2b61-8eca-4ddc-bf00-ba2891aff5ef
To: /data.zip
100%|██████████| 156M/156M [00:04<00:00, 35.1MB/s] 


['Data']


In [48]:
import os
import cv2
import sklearn
import tensorflow as tf
from tensorflow.keras import layers
import numpy as np
DATA_DIR = "data/Data"
IMG_SIZE = 224
CLASSES = ["glioma", "meningioma", "notumor", "pituitary"]
VALID_EXT = {".jpeg", ".jpg", ".bmp", ".png"}

In [49]:
def scan_dataset(data_dir=DATA_DIR, classes=CLASSES, valid_ext=VALID_EXT):
    kept, rejected = [], []

    for label in classes:
        folder = os.path.join(data_dir, label) 

        for filename in os.listdir(folder):
            path = os.path.join(folder, filename)
            if os.path.splitext(filename)[1].lower() not in valid_ext:
                rejected.append((path, "bad extension"))
                continue

            try:
                img = cv2.imread(path)
                if img is None:
                    rejected.append((path, "unreadable"))
                else:
                    kept.append((path, label))
            except Exception as e:
                rejected.append((path, str(e)))
    return kept, rejected


def load_image(path, size=IMG_SIZE):
    img = cv2.imread(path)
    if img is None:
        raise ValueError(f"Unreadable image: {path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (size, size), interpolation=cv2.INTER_AREA)


def load_dataset(kept, classes=CLASSES, size=IMG_SIZE, seed=42):
    X = np.empty((len(kept), size, size, 3), dtype=np.uint8)
    y = np.empty(len(kept), dtype=np.int64)
    class_to_idx = {c: i for i, c in enumerate(classes)}

    for i, (path, label) in enumerate(kept):
        X[i] = load_image(path, size)
        y[i] = class_to_idx[label]

    idx = np.random.default_rng(seed).permutation(len(X))
    return X[idx], y[idx]

In [55]:
kept , rejected = scan_dataset()
X, y = load_dataset(kept)
y

array([1, 2, 3, ..., 3, 3, 1])

In [51]:
data_augmentation = tf.keras.Sequential([
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
], name = 'data_augmentation')

In [52]:
from sklearn.model_selection import train_test_split

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.15, stratify=y_tmp, random_state=42)

for name, labels in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(name, len(labels), np.bincount(labels))

train 5073 [1171 1188 1445 1269]
val 896 [207 210 255 224]
test 1054 [243 247 300 264]


In [53]:
X_train = X_train.astype("float32") / 255.0
X_val   = X_val.astype("float32") / 255.0
X_test  = X_test.astype("float32") / 255.0

In [54]:
AUTOTUNE = tf.data.AUTOTUNE
BATCH = 32

train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .cache()
            .shuffle(len(X_train))
            .map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=AUTOTUNE)
            .batch(BATCH)
            .prefetch(AUTOTUNE))

val_ds  = tf.data.Dataset.from_tensor_slices((X_val, y_val)).cache().batch(BATCH).prefetch(AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH).prefetch(AUTOTUNE)

KeyboardInterrupt: 